In [ ]:
import os
import gc
import tensorflow as tf
from tqdm import tqdm
import numpy as np
import pandas as pd
import random
from tensorflow.keras.layers import Dropout, Dense, Average, GlobalAveragePooling3D
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, CSVLogger
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import sys
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
tf.keras.mixed_precision.set_global_policy('mixed_float16')

sys.path.insert(0, '/kaggle/input/datasets/ganuwoahh/i3d-no-top-4') # i3d on keras without the classification layer
from i3d_inception import Inception_Inflated3d

TFRECORD_BASE = '/kaggle/input/datasets/ganuwoahh/cholec80-tfrecords-real/tfrecords' 

train_files = [os.path.join(TFRECORD_BASE, f'video{i:02d}.tfrecord') for i in range(1, 71)]
val_files = [os.path.join(TFRECORD_BASE, f'video{i:02d}.tfrecord') for i in range(71, 81)] # no test set here because needed more for training already not a big dataset

In [ ]:
def parse_tfrecord(example): # parsing the .tfrecord
    feature_description = {
        'rgb_image': tf.io.FixedLenFeature([], tf.string),
        'flow_image': tf.io.FixedLenFeature([], tf.string),
        'label': tf.io.FixedLenFeature([], tf.int64),
    }
    parsed = tf.io.parse_single_example(example, feature_description)
    
    rgb = tf.image.decode_jpeg(parsed['rgb_image'], channels=3)
    rgb = tf.cast(rgb, tf.float32) / 127.5 - 1.0
    
    flow = tf.cond(
        tf.equal(tf.strings.length(parsed['flow_image']), 0), # an ifelse statement that detects the first image without an optical flow and generates a blank one which is what you want
        lambda: tf.ones([224, 224, 3], dtype=tf.float32) * 128.0, 
        lambda: tf.cast(tf.image.decode_jpeg(parsed['flow_image'], channels=3), tf.float32)
    )

    flow = (flow[:, :, :2] - 128.0) / 128.0 # deleting the red layer because we hard coded it to be 128 anyway so don't need to load it for each frame
    
    return {'rgb': rgb, 'flow': flow}, parsed['label'] # the three things we packed together

In [ ]:
def process_single_video(filepath): # making a video out of the frames we parsed
    ds = tf.data.TFRecordDataset(filepath, num_parallel_reads=1)
    ds = ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    
    ds = ds.window(size=32, shift=8, drop_remainder=True) # size is how many frames together. so 32 is 32 seconds of footage. increase if your hardware can do it
    ds = ds.flat_map(lambda x, y: tf.data.Dataset.zip(( # shift is the stride. if stride = size then you have no overlap and will miss information. instead of [0-31][32-63] we get [0-31][8-39]
        {
            'rgb': x['rgb'].batch(32), 
            'flow': x['flow'].batch(32)
        }, 
        y.batch(32) # stacking a bunch of frames on top of each other to make a 4d vector
    )))
    return ds

In [ ]:
def build_dataset(tfrecord_files, batch_size=8, training=True): # added interleaving because hardware limits i couldn't load enough to the ram at once
    ds = tf.data.Dataset.from_tensor_slices(tfrecord_files)
    
    if training:
        ds = ds.shuffle(len(tfrecord_files)) # shuffle the file paths
        
    ds = ds.interleave( # basically open 8 random videos and take 1 clip from each
        process_single_video,
        cycle_length=8 if training else 2,
        block_length=1,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=not training
    )
    
    ds = ds.map(lambda inputs, labels: ( # predict the phase you think it's in at the very last frame and one hot encode
        (inputs['rgb'], inputs['flow']), 
        tf.one_hot(labels[-1], depth=7)
    ), num_parallel_calls=tf.data.AUTOTUNE)
    
    if training:
        # 4. Tiny Shuffle Buffer! 
        # Since we are interleaving 8 videos, a buffer of 50 is more than enough. RAM is saved.
        ds = ds.shuffle(buffer_size=50) # IF YOUR HARDWARE CAN MANAGE JUST IGNORE THE INTERLEAVING AND MAKE THIS LIKE 5000 OR 10000
        ds = ds.repeat() 
        
    ds = ds.batch(batch_size, drop_remainder=True) # drop remainder prevents uneven batches
    ds = ds.prefetch(tf.data.AUTOTUNE) 
    return ds

train_dataset = build_dataset(train_files)
val_dataset = build_dataset(val_files, training=False)

In [ ]:
strategy = tf.distribute.MirroredStrategy() # using dual NVIDIA T4s

"""
    basically i stillll kept running out of ram after like 5 or so epochs so i saved the model each epoch and just reloaded it if the VM crashed
    so if you don't have that issue, just trim out most of the load save and reload fluff
"""

CHECKPOINT_PATH = '/kaggle/working/best_i3d_cholec80.keras'

with strategy.scope():
    if os.path.exists(CHECKPOINT_PATH):
        print(f"\n found {CHECKPOINT_PATH}")
        model = load_model(CHECKPOINT_PATH)
        
        if os.path.exists('/kaggle/working/training_history.csv'):
            df = pd.read_csv('/kaggle/working/training_history.csv')
            INITIAL_EPOCH = df['epoch'].max()
            print(f"[+] resuming from epoch {INITIAL_EPOCH}...")
        else:
            INITIAL_EPOCH = 0
            
    else:
        INITIAL_EPOCH = 0
        
        rgb_base = Inception_Inflated3d(include_top=False, weights='rgb_kinetics_only', input_shape=(32, 224, 224, 3))
        rgb_base.trainable = False  # froze backbone
        for layer in rgb_base.layers: layer._name, layer.name = layer.name + '_rgb', layer.name + '_rgb'
        
        flow_base = Inception_Inflated3d(include_top=False, weights='flow_kinetics_only', input_shape=(32, 224, 224, 2))
        flow_base.trainable = False  # froze backbone
        for layer in flow_base.layers: layer._name, layer.name = layer.name + '_flow', layer.name + '_flow'

        rgb_pool = GlobalAveragePooling3D()(rgb_base.output)
        flow_pool = GlobalAveragePooling3D()(flow_base.output)

        combined = Average()([rgb_pool, flow_pool])
        x = Dropout(0.5)(combined)
        
        output = Dense(7, activation='softmax', name='cholec80_phases', dtype='float32')(x) # put the last layer back into 32 bit
        
        model = Model(inputs=[rgb_base.input, flow_base.input], outputs=output)
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
            loss='categorical_crossentropy',
            metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), # these metrics don't do much here.
                     tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
        )

        """
            accuracy doesn't account for class imbalance. precision and recall have an inbuilt threshold of 0.5 so unless realllllly confident it doesn't accurately capture the model. so we're 
            checkpointing on auc here.
        """

In [ ]:
class MemoryCleanupCallback(tf.keras.callbacks.Callback): # manually clear ram after each epoch because turns out tensorflow doesn't do that for you
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()
        tf.keras.backend.clear_session()

cholec80_weights = {0: 3.28, 1: 0.33, 2: 1.68, 3: 0.51, 4: 3.32, 5: 1.71, 6: 3.72} # weightage based on how often they show up because limited training data

callbacks = [
    ModelCheckpoint('/kaggle/working/best_i3d_cholec80.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    CSVLogger('/kaggle/working/training_history.csv', append=True),
    MemoryCleanupCallback()
]

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    initial_epoch=INITIAL_EPOCH,
    epochs=20, # 12 hour limit on vm. maybe 25-30 would be sweet spot but who knows
    steps_per_epoch=2139, # length of the largest video in the dataset / 8 (batchsize)
    callbacks=callbacks,
    class_weight=cholec80_weights
)

In [ ]:
model.save('/kaggle/working/final_state_i3d_cholec80.keras')
print("\nTraining Complete and Final Model Saved.")

In [ ]:
phase_names = [
    "0: Preparation", "1: Calot Triangle Dissection", "2: Clipping & Cutting",
    "3: Gallbladder Dissection", "4: Gallbladder Packaging", "5: Cleaning & Coagulation", "6:Gallbladder Retrieval"
]

model = load_model('/kaggle/working/final_state_i3d_cholec80.keras')

y_true = []
y_pred_probs = []

for (rgb, flow), labels in tqdm(val_dataset, desc="Evaluating Batches"):
    preds = model.predict_on_batch([rgb, flow])
    y_pred_probs.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)

y_true_classes = np.argmax(y_true, axis=1)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

report = classification_report(y_true_classes, y_pred_classes, target_names=phase_names, zero_division=0) # we just argmax first instead of whatever default keras is doing
print(report)

In [ ]:
cm = confusion_matrix(y_true_classes, y_pred_classes)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_normalized = np.nan_to_num(cm_normalized)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues", 
            xticklabels=phase_names, yticklabels=phase_names)
plt.title("Normalized Confusion Matrix")
plt.ylabel("True Surgical Phase")
plt.xlabel("Predicted Surgical Phase")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink'] # choose your colours :D

for i in range(7):
    if np.sum(y_true[:, i]) > 0:
        fpr, tpr, _ = roc_curve(y_true[:, i], y_pred_probs[:, i])
        roc_auc = auc(fpr, tpr)
        
        plt.plot(fpr, tpr, color=colors[i], lw=2,
                 label=f'{phase_names[i]} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Class ROC Curves')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curves.png', dpi=300)
plt.show()